<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/11_plugins.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.6.0?labpath=examples%2F11_plugins.ipynb)

# 11 — Plugin System

Register custom encoders and exporters via the plugin registry.
Use them end-to-end with `prepare()`, the CLI, and all built-in exporters — no fork required.

| Registry function | Purpose |
|---|---|
| `register_encoder(name)` | Decorate a `BaseEncoder` subclass to make it usable by name |
| `register_exporter(name)` | Decorate an exporter class to wire it into `prepare(framework=...)` |
| `list_encoders()` | List all custom-registered encoder names |
| `list_exporters()` | List all custom-registered exporter names |
| `unregister_encoder(name)` | Remove an encoder (useful in tests) |
| `unregister_exporter(name)` | Remove an exporter (useful in tests) |

In [ ]:
import numpy as np

import quprep as qd
from quprep.encode.base import BaseEncoder, EncodedResult
from quprep.plugins import (
    list_encoders,
    list_exporters,
    register_encoder,
    register_exporter,
    unregister_encoder,
    unregister_exporter,
)

rng = np.random.default_rng(42)
data = rng.uniform(0, 1, size=(8, 3))

## 1. Custom encoder

Decorate any `BaseEncoder` subclass with `@register_encoder("name")`. Set `metadata["encoding"]` to a built-in encoding name to reuse that encoding's QASM/Braket/Q#/IQM export path.

In [ ]:
@register_encoder("sigmoid_angle")
class SigmoidAngleEncoder(BaseEncoder):
    """Features encoded as sigmoid-scaled Ry angles in (0, π)."""

    @property
    def n_qubits(self):
        return None  # data-dependent

    @property
    def depth(self):
        return 1

    def encode(self, x):
        angles = np.pi / (1.0 + np.exp(-x))
        return EncodedResult(
            parameters=angles,
            metadata={"encoding": "angle", "rotation": "ry", "n_qubits": len(x), "depth": 1},
        )


result = qd.prepare(data, encoding="sigmoid_angle", framework="qasm")
print(f"Encoding : {result.encoded[0].metadata['encoding']}")
print(f"n_qubits : {result.encoded[0].metadata['n_qubits']}")
print(f"Circuits : {len(result.circuits)}")
print()
print("First QASM circuit:")
print(result.circuit)

## 2. Another encoder — DCT angle

Any transform that produces angle-like values works as a drop-in encoder.

In [ ]:
@register_encoder("fourier_angle")
class FourierAngleEncoder(BaseEncoder):
    """Encode features via their discrete cosine transform."""

    @property
    def n_qubits(self):
        return None

    @property
    def depth(self):
        return 1

    def encode(self, x):
        dct = np.fft.rfft(x, norm="ortho").real
        angles = np.clip(dct, -np.pi, np.pi)
        return EncodedResult(
            parameters=angles,
            metadata={"encoding": "angle", "rotation": "rz", "n_qubits": len(angles), "depth": 1},
        )


x = rng.uniform(-1, 1, 3)
r = FourierAngleEncoder().encode(x)
print(f"Input  : {x.round(4)}")
print(f"Angles : {r.parameters.round(4)}")

## 3. List registered plugins

`list_encoders()` / `list_exporters()` return only custom-registered names.

In [ ]:
print(f"Encoders  : {list_encoders()}")
print(f"Exporters : {list_exporters()}")

## 4. Custom exporter

`@register_exporter("name")` wires the class into `prepare(framework="name")`.

In [ ]:
@register_exporter("json_angles")
class JsonAnglesExporter:
    """Export encoded circuit as a JSON-serialisable dict of angles."""

    def export(self, encoded: EncodedResult) -> dict:
        return {
            "encoding": encoded.metadata.get("encoding"),
            "n_qubits": encoded.metadata.get("n_qubits"),
            "angles": encoded.parameters.tolist(),
        }

    def export_batch(self, encoded_list):
        return [self.export(e) for e in encoded_list]


result_json = qd.prepare(data, encoding="angle", framework="json_angles")
print(f"Type    : {type(result_json.circuits[0])}")
print(f"Circuit : {result_json.circuits[0]}")

## 5. Plugin encoder + plugin exporter together

In [ ]:
result_combo = qd.prepare(data, encoding="sigmoid_angle", framework="json_angles")
first = result_combo.circuits[0]
print(f"Encoding : {first['encoding']}")
print(f"n_qubits : {first['n_qubits']}")
print(f"Angles   : {[round(a, 4) for a in first['angles']]}")

## 6. Unregister

Clean up when done — essential in tests to avoid state leaking between test cases.

In [ ]:
unregister_encoder("sigmoid_angle")
unregister_encoder("fourier_angle")
unregister_exporter("json_angles")

print(f"Encoders after cleanup  : {list_encoders()}")
print(f"Exporters after cleanup : {list_exporters()}")